# Lesson 3: Wrap a PyTorch model

**You will learn:** how to turn a PyTorch network into a subject with `BrainScoreModel`, map brain regions to its layers, and record them.
**Requires:** CPU only, under a minute. The network and images are committed with the lessons.

**The question:** in a network trained on ImageNet, is the activity of its last convolutional layer more alike for two images of the same CIFAR-10 category than for images of different categories? The network was trained to name 1,000 ImageNet classes, not these ten categories, and these images come from a different dataset.

## Setup

Imports and settings that are not part of the method. The cell is folded; click it to expand.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. The **extractor** prepares images and records layers; the **region map** says which layer stands for which brain area (a hypothesis, not a fact).

`VisionWrapper` is the extractor for image models: it reads the image files, prepares them with `preprocessing` (resizing and normalizing, defined in Setup), passes them through the network and records the layers the region map names.

In [2]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'})

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(subject.out_channels)                          # what this subject can produce

['neural:IT', 'neural:V1', 'neural:V4']

## The experiment

### Stimuli

A `StimulusSet` is a table with one row per stimulus: an identifier (`stimulus_id`), the image file and any labels. Every response the subject gives is linked back to its row by `stimulus_id`.

In [3]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

### Task

Plain viewing: each image is shown once and two regions, V4 and IT, are recorded. There is no answer to give.

## Run

A **session** is one visit by the subject. `BufferedSession` holds what to present (`stimulus_set`) and what the experiment asks for (`requested_output_channels`, here the activity of two regions). `subject.participate(session)` runs the subject through it, and `session.collect(channel)` returns what was recorded. Both regions come from one pass through the network.

In [4]:
from brainscore_core.streaming import BufferedSession

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:V4', 'neural:IT'])
subject.participate(session)
v4 = session.collect('neural:V4')
it = session.collect('neural:IT')

`participate(session)` is how benchmarks and experiments run a subject. `process(stimuli)` is a typed shortcut that gives the same numbers; use it interactively, not in benchmarks.

In [5]:
subject.start_recording('IT')
shortcut = subject.process(stimuli)
np.allclose(shortcut.values, it.values)

True

## Read results back

Each recording has one row per image and one column per unit of that layer.

In [6]:
print('V4', v4.shape, ' IT', it.shape, '(images x units)')

V4 (120, 93312)  IT (120, 86528) (images x units)


## Analysis

Plain Python from here on: no Brain-Score calls. Correlate every pair of images' activity, then compare pairs from the same category with pairs from different categories. The null shuffles the category labels 1,000 times.

In [7]:
def category_gap(similarity, labels):
    """Mean similarity of same-category pairs minus that of different-category pairs."""
    upper = np.triu_indices(len(labels), 1)                   # each pair of images once
    same = (labels[:, None] == labels[None, :])[upper]
    return similarity[upper][same].mean() - similarity[upper][~same].mean()

def shuffled_gaps(similarity, labels, n=1000, seed=0):
    """The gap with the category labels shuffled, n times: the null."""
    rng = np.random.default_rng(seed)                         # fixed seed: same shuffles every run
    return np.array([category_gap(similarity, rng.permutation(labels)) for _ in range(n)])

## Results

In [8]:
similarity = np.corrcoef(np.asarray(it.values))               # image-by-image correlation of IT activity
labels = np.asarray(stimuli['category'])
observed = category_gap(similarity, labels)
null = shuffled_gaps(similarity, labels)
print(f'same minus different: {observed:.3f}; shuffled labels: {null.mean():.3f} +/- {null.std():.3f}; '
      f'p = {np.mean(null >= observed):.3f}')

same minus different: 0.032; shuffled labels: -0.000 +/- 0.001; p = 0.000


If the observed gap sits far above the shuffled ones, the trained layer groups these images by category although it learned on different images and labels. Lesson 5 asks how much of that comes from training at all, using the same network with random weights.

## Summary

- `BrainScoreModel(model=..., extractor={...}, region_layer_map={...})` makes a network a subject.
- A session asks for `neural:<region>`; several regions come from one pass.
- Every result needs a null: here, shuffled labels.

**Next:** [Lesson 4](04_bring_an_existing_model.ipynb) brings in models registered for classic Brain-Score.